In [1]:
#import relevant libraries
import os
#from scipy import stats

#import scipy as sp
import pandas as pd
import dabest
import NLCLIMB
from datetime import datetime
date = datetime.today().strftime('%Y%m%d')



#NOTE: SUPPRESSES WARNINGS!

import warnings


warnings.simplefilter(action="ignore", category=RuntimeWarning)
warnings.simplefilter(action="ignore", category=UserWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)


Pre-compiling numba functions for DABEST...


Compiling numba functions: 100%|██████████| 11/11 [00:00<00:00, 19.44it/s]


Numba compilation complete!


In [2]:
#initial file processing
workcomp = "C:\\Users\\User"
laptop = "C:\\Users\\lnico"
officecomp = "C:\\Users\\Star"
homecomp = "D:"
specifiedpath = homecomp


filedir = "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\Climbing_New\\"
savedir = specifiedpath + filedir + "Compilation with delta\\2025deltagcollection\\"
saveosardir = specifiedpath + filedir + "Compilation with delta\\2025fallingtoosarcomp\\"
    
openPath = specifiedpath + filedir
files = os.listdir(openPath)

#identifying genotypes
responder = "ACR"
respondercsv = responder + ".csv"
wt = "w1118"


In [5]:
lstnew=[]

#lstnew should be the list of names you want to process the files with. Only choose one

#if you want to process all the names in the filedir

for file_no in os.listdir(openPath): 
    if respondercsv in file_no and "w1118" not in file_no :   
        f = os.path.join(openPath, file_no)
        dfe=pd.read_csv(f)
        exptdf = dfe.drop(dfe.columns[[0]],axis = 1)
        driver = file_no.split(" ")[0]
        lstnew.append(driver)
#lst = lstnew.copy()
lst = [x for x in lstnew if x not in ['Th-Gal4', 'R58']]

#processing ONLY specific names
# lst = ["MB112C"]
lst = ['MB319C', 'SS67662', 'SS86947', 'SS01308']
print(lst)

['MB319C', 'SS67662', 'SS86947', 'SS01308']


In [6]:
for n in lst:
    driver = n
    print(n)
    transgenic = driver + " x " + responder
    filename = openPath + transgenic + ".csv"
    filenamewt = openPath + wt+"_"+ transgenic + ".csv"

    dfe=pd.read_csv(filename)
    dfw= pd.read_csv(filenamewt)

    exptdf = dfe.drop(dfe.columns[[0]],axis = 1)
    wtdf = dfw.drop(dfw.columns[[0]],axis = 1)

    #adjust this depending on timeframe
    dfexpt = NLCLIMB.timerule(NLCLIMB.generation(exptdf, driver))
    dfwt = NLCLIMB.timerule(NLCLIMB.generation(wtdf, wt))
       
    #processing before dabest application 
    df_f = NLCLIMB.fallingocc(dfexpt, dfwt).reset_index(drop=True)
    df_sp = NLCLIMB.ospeed(dfwt, dfexpt).reset_index(drop=True)
    df_bsp = NLCLIMB.bspeed(NLCLIMB.boutspeed(dfexpt), NLCLIMB.boutspeed(dfwt)).reset_index(drop=True)
    df_h = NLCLIMB.totalheight(dfexpt, dfwt).reset_index(drop=True)
    df_pp = NLCLIMB.bheight(NLCLIMB.pauseheight(dfexpt), NLCLIMB.pauseheight(dfwt)).reset_index(drop=True)
    df_maxv = pd.concat([NLCLIMB.maxvelocity(dfexpt, "Expt"), NLCLIMB.maxvelocity(dfwt, "WT")], axis = 0).reset_index(drop=False)
    df_sim = pd.concat([NLCLIMB.straightnessindexmeter(dfexpt, "Expt"), NLCLIMB.straightnessindexmeter(dfwt, "WT")], axis = 0).reset_index(drop=True)

    #pause and bouts
    wttotalmeanevent, wttotalnumberevent = NLCLIMB.pausecomp(dfwt, wt)
    expttotalmeanevent, expttotalnumberevent = NLCLIMB.pausecomp(dfexpt, driver)
    alltgtmeandf_bout = pd.concat([NLCLIMB.pausenumber(wttotalmeanevent, n, "Bouts"), NLCLIMB.pausenumber(expttotalmeanevent, n, "Bouts")], axis = 0).reset_index(drop=True)
    alltgtnumberdf_bout = pd.concat([NLCLIMB.pausenumber(wttotalnumberevent, n, "Bouts"), NLCLIMB.pausenumber(expttotalnumberevent, n, "Bouts")], axis = 0).reset_index(drop=True)
            
    #___________________________________________#    
    # meandiff plots -- you run mean_diff instead of delta_g because since all the binary data is at the same dimension, no standardization is required and empirical delta delta is sufficient
    #dff2_prop = NLCLIMB.deltaversion_meandiff(df_f, "binary_fallvalue", "fallprop")
    dff2_number = NLCLIMB.deltaversion_meandiff(df_f, "Fall", "fallnumber")   #number of falls is under deltaversion_binary because the number of flies that fall could actually be so few in number, that the SD is 0, and thus hedges g will not be able to perform since the divisor ==0

    #deltag plots
    dfs2 = NLCLIMB.deltaversion_deltag(df_sp, "Velocity", "speed")
    dfh2 = NLCLIMB.deltaversion_deltag(df_h, "Y", "height")
    dfbs2 = NLCLIMB.deltaversion_deltag(df_bsp, "BSpeed", "bspeed")
    dfpp2 = NLCLIMB.deltaversion_deltag(df_pp, "Height", "pausepos")
    dfmv2 = NLCLIMB.deltaversion_deltag(df_maxv, "maxvelocity", "maxvelocity")
    dfsim2 = NLCLIMB.deltaversion_deltag(df_sim, "averagestraightnessindex", "straightindex")
    #pause and bouts
    dfmb2 = NLCLIMB.deltaversion_deltag(alltgtmeandf_bout, "Bouts", "meanbout")     
    dfnb2 = NLCLIMB.deltaversion_deltag(alltgtnumberdf_bout, "Bouts", "bout")
    
    #singledelta processing
    lsr_bsp = NLCLIMB.log2metric(df_bsp, "BSpeed")
    lsr_sp = NLCLIMB.log2metric(df_sp, 'Velocity')    
    
    #new index and ratio metrics
    bout_index_nb = NLCLIMB.boutindex(alltgtnumberdf_bout, "Bouts") 
    ratio_mb = NLCLIMB.simplemetricratio(alltgtmeandf_bout, "Bouts") 
    ratio_maxv = NLCLIMB.simplemetricratio(df_maxv, "maxvelocity")
    

    df_lsrbsp = NLCLIMB.singledelta(lsr_bsp, "log2 BSpeed", "log2bspeed")
    df_lsrsp = NLCLIMB.singledelta(lsr_sp, "log2 Velocity", "log2speed")
    df_boutindex_nb = NLCLIMB.singledelta(bout_index_nb, "Bouts", "boutnumber_index")  # Renamed from ratio to index
    df_ratio_mb = NLCLIMB.singledelta(ratio_mb, "Bouts", "boutduration_ratio")
    df_ratio_maxv = NLCLIMB.singledelta(ratio_maxv, "maxvelocity", "maxvelocity_ratio")
    
    #final df and saving into excel
    dftotal = pd.concat([dff2_number, dfs2, dfh2, dfbs2, dfpp2, dfmv2, dfsim2, dfmb2, dfnb2, df_lsrbsp, df_lsrsp, df_boutindex_nb, df_ratio_mb, df_ratio_maxv], axis = 1)
    dftotal['MBON'] = n
    dftotal.set_index("MBON", inplace = True)
    #dftotal.to_csv(savedir + n + " x " + responder + "_deltag_allstats.csv")
    
print("Done!")        

MB319C


KeyboardInterrupt: 